# Survival Analysis: Time to Defibrillation vs. Patient Outcomes (Unbinned Logistic Regression)

This notebook analyzes the Out-of-Hospital Cardiac Arrest (OHCA) dataset to quantify how delay to first defibrillation affects patient outcomes.

Specifically, it uses **unbinned patient-level data** (one row per patient), computes **Time to First Defibrillation** from dispatch-call and first-shock timestamps, and fits two **logistic regression models** (`statsmodels.Logit`) over a constrained 0–20 minute window:

- **Predictor (X):** Time to First Defibrillation (minutes)
- **Outcome 1 (y):** Survival Status (1 = Survived, 0 = Dead)
- **Outcome 2 (y):** CPC Status (1 = Good CPC 1–2, 0 = Poor CPC 3–5)

Key outputs include:

- Logistic regression coefficient estimates, p-values, and model summaries
- **Odds Ratio** interpretation for each 1-minute delay (Survival and CPC)
- Fitted probability curves for Survival and CPC across 0–20 minutes
- A combined visualization comparing both modeled outcome curves

This provides a standard epidemiological estimate of the time–outcome relationship without time binning or exponential curve fitting.

# Need to install decryption Libraries

In [ ]:
%pip install msoffcrypto-tool openpyxl

# Install libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import msoffcrypto
import io
import openpyxl
import statsmodels.api as sm
from scipy.optimize import curve_fit

from pathlib import Path

# Get the password from env

In [2]:
password = os.getenv("PAROS_PASSWORD")

if not password:
    print("ERROR: Password not found in the env")
else:
    print("SUCCESS: Password found in the env")

SUCCESS: Password found in the env


# Setting up file paths

In [3]:
CURRENT_DIRECTORY = Path(os.getcwd())
BASE_DATASET_PATH = CURRENT_DIRECTORY.parents[0] / "datasets"
ENCRYPTED_FILE_PATH = BASE_DATASET_PATH / "DataExportParos_SG_Apr10-Dec21_with postal codes_27Mar25_encryptedv1.xlsx"

display(ENCRYPTED_FILE_PATH)


PosixPath('/Users/axlee/Desktop/Singhealth/AED-OHCA/datasets/DataExportParos_SG_Apr10-Dec21_with postal codes_27Mar25_encryptedv1.xlsx')

# Decrypting the file

In [4]:
decrypted_workbook = io.BytesIO()
print(f"Attempting to decrypt: {ENCRYPTED_FILE_PATH.name}...")

try:
    with open(ENCRYPTED_FILE_PATH, 'rb') as file:
        office_file = msoffcrypto.OfficeFile(file)
        office_file.load_key(password=password)
        office_file.decrypt(decrypted_workbook)

    # Load the decrypted memory object directly into a Pandas DataFrame
    df = pd.read_excel(decrypted_workbook)
    print("✅ RAW PAROS dataset successfully decrypted and loaded!")
    
    # Show the first 3 rows to confirm
    # display(df.head(3))

except FileNotFoundError:
    print(f"❌ Error: Could not find the file at {ENCRYPTED_FILE_PATH}. Please check the path and filename.")
except openpyxl.utils.exceptions.InvalidFileException:
    print("❌ Error: Invalid password or unsupported Excel format.")
except Exception as e:
    print(f"❌ An error occurred: {e}")

Attempting to decrypt: DataExportParos_SG_Apr10-Dec21_with postal codes_27Mar25_encryptedv1.xlsx...


/opt/miniconda3/envs/geospatial_env/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


✅ RAW PAROS dataset successfully decrypted and loaded!


In [5]:
print(df.columns.tolist())


['Case #', 'Country', 'City', 'Site #', 'Patient brought in by', 'Date of Incident', 'Location of incident', 'Location Unknown', 'Location Type', 'Location Type Other', 'Age', 'Age Modifier', 'Gender', 'Race', 'Medical History - No', 'Medical History - Unknown', 'Medical History - Heart disease', 'Medical History - Diabetes', 'Medical History - Cancer', 'Medical History - Hypertension', 'Medical History - Renal Disease', 'Medical History - Respiratory Disease', 'Medical History - Hyperlipidemia', 'Medical History - Stroke', 'Medical History - HIV', 'Medical History - Other', 'Time call received at dispatch center', 'No First Responder dispatched', 'Time First Responder dispatched', 'Time Ambulance dispatched', 'Time First Responder arrived at scene', 'Time Ambulance arrived at scene', 'Time EMS arrived at patient side', 'Time Ambulance left scene', 'Time Ambulance arrived at ED', 'Estimated time of arrest', 'Estimated time of arrest unknown', 'Arrest witnessed by', 'Bystander CPR', 'DA

# Cleaning Paros Datset

In [6]:
# strip whitespace from all string cells first
df = df.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))

missing_tokens = [
    "", " ", "  ", "N/A", "n/a", "NA", "na", "NULL", "null", "None", "none", "-"
]
df = df.replace(missing_tokens, np.nan)

missing_summary = (
    df.isna().mean()
      .mul(100)
      .sort_values(ascending=False)
      .rename("missing_pct")
      .to_frame()
)
display(missing_summary.head(30))
display(df.head(3))

/var/folders/y3/l8mcqj111md_2kpwgj8zhk0w0000gn/T/ipykernel_96278/3896428710.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(missing_tokens, np.nan)


,missing_pct
Lidocaine,99.996511
Bicarbonate,99.989534
Amiodarone,99.975580
Patient neurological status - Unknown,99.972091
EQ-5D Anxiety/Depression,99.958137
EQ-5D Pain/Discomfort,99.951160
Atropine,99.916274
EQ-5D Unknown,99.905808
Defibrillation performed by - Bystander - Family,99.902320
Dextrose,99.881388


,Case #,Country,City,Site #,Patient brought in by,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,...,EQ-5D Unknown,EQ-5D Mobility,EQ-5D Self-care,EQ-5D Usual activities,EQ-5D Pain/Discomfort,EQ-5D Anxiety/Depression,EQ-5D VAS,General Comments,Date Created,Date Last Saved
0,SGSIN0213,SG,SIN,2,EMS,2010-04-01,470146.0,NaN,Home Residence,HDB Level 7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,2011-02-22
1,SGSIN0218,SG,SIN,2,EMS,2010-04-01,520926.0,NaN,Home Residence,HDB Level 2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,2011-02-22
2,SGSIN6480,SG,SIN,6,EMS,2010-04-01,560565.0,NaN,Healthcare Facility,NKF Dialysis Centre,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,2012-04-18


In [7]:
df['Cause of arrest'].value_counts(dropna=False)

Cause of arrest
NaN           27480
Non-Trauma     1185
Name: count, dtype: int64

# Setting columns for feature engineering

In [10]:
# This will be used to calculate time to defib
call_time_col = 'Time call received at dispatch center'
shock_time_col = 'Time of first shock given'
ems_cpr_time_col = 'Time CPR started by EMS/Private ambulance'

# This will be used to calculate into a binary status of 0/1 survival
status_col = 'Patient status'

df_survival = df.copy()
# display(df_survival[[call_time_col, shock_time_col, status_col]].head())

# Calculating the time to defib

In [11]:
df_survival["Call_Time"] = pd.to_datetime(df_survival[call_time_col].astype(str),
                                          format = "mixed",
                                          errors='coerce')
df_survival['Shock_Time'] = pd.to_datetime(df_survival[shock_time_col].astype(str),
                                           format='mixed',
                                           errors='coerce')

# Calculate the difference in minutes
df_survival['Time_to_Defib'] = (df_survival['Shock_Time'] - df_survival['Call_Time']).dt.total_seconds() / 60

# Calculate time to EMS CPR as well
df_survival['EMS_CPR_Time'] = pd.to_datetime(
    df_survival[ems_cpr_time_col].astype(str),
    format='mixed',
    errors='coerce'
)

df_survival['Time_from_Call_to_EMS_CPR'] = (
    df_survival['EMS_CPR_Time'] - df_survival['Call_Time']
).dt.total_seconds() / 60


# Fix midnight crossover overlaps
df_survival.loc[df_survival['Time_to_Defib'] < 0, 'Time_to_Defib'] += 1440
df_survival.loc[df_survival['Time_from_Call_to_EMS_CPR'] < 0, 'Time_from_Call_to_EMS_CPR'] += 1440


# display(df_survival[['Call_Time', 'Shock_Time', 'Time_to_Defib']].head(10))

Time to defib will show 13.9 minutes (standard decimal format)

Example (Index 0):
-   54 seconds ÷ 60 seconds = 0.90 minutes.
-   13 minutes + 0.90 minutes = 13.90 minutes.

# Feature Engineering for Survival Status

- 0 = Dead
- 1 = Survived

In [ ]:
df_survival['Survival_Status'] = df_survival[status_col].astype(str).str.contains('Discharged|Alive|Remains', 
                                                                                  case=False, 
                                                                                  na=False
).astype(int)
df_survival['Survival_Status'] = df_survival['Survival_Status'].astype(int)

# display(df_survival[['Time_to_Defib', 'Survival_Status']].head())

# Filter to a standard range (e.g., 0-20 minutes)
- This is to avoid extreme outliers skewing the fit

In [ ]:
df_plot = df_survival[(df_survival['Time_to_Defib'] >= 0) & (df_survival['Time_to_Defib'] <= 20)].copy()


# Define the Piecewise Linear Regression Model (Change-point Model)
- It features an initial slope (b1) that switches to a second slope (b2) at time 'tau'

In [ ]:
def piecewise_linear(x, tau, b0, b1, b2):
    return np.where(x < tau, 
                    b0 + b1 * x, 
                    b0 + b1 * tau + b2 * (x - tau))

# Standardize data to the 0-20 min window
df_analysis = df_survival[(df_survival['Time_to_Defib'] >= 0) & (df_survival['Time_to_Defib'] <= 20)].copy()

# Setup arrays to store results
bin_labels = ['Continuous\n(Raw Data)']
estimated_taus = []

# Fit the Continuous (Unbinned) Baseline

In [ ]:
x_cont = df_analysis['Time_to_Defib'].values
y_cont = df_analysis['Survival_Status'].values

# Initial guesses: Tau=6 min, int=0.5, slope1=-0.05, slope2=0
try:
    popt_cont, _ = curve_fit(piecewise_linear, x_cont, y_cont, 
                             p0=[6.0, 0.5, -0.05, 0.0], 
                             bounds=([1, 0, -1, -1], [18, 1, 1, 1]))
    estimated_taus.append(popt_cont[0])
except Exception as e:
    print(f"Continuous fit failed: {e}")
    estimated_taus.append(np.nan)

# Fit the Binned Models (2-min, 3-min, 5-min)

In [ ]:
bin_sizes = [2, 3, 5]

for b in bin_sizes:
    bin_labels.append(f'{b}-min Bins')
    
    # Bin the time based on size 'b'
    df_analysis[f'Bin_{b}'] = np.floor(df_analysis['Time_to_Defib'] / b) * b
    
    # Calculate survival probability per bin
    binned = df_analysis.groupby(f'Bin_{b}').agg(
        Survival_Probability=('Survival_Status', 'mean')
    ).reset_index()
    
    # Use the midpoint of the bin for an accurate spatial representation
    x_binned = binned[f'Bin_{b}'] + (b / 2.0)
    y_binned = binned['Survival_Probability']
    
    try:
        popt_binned, _ = curve_fit(piecewise_linear, x_binned, y_binned, 
                                   p0=[6.0, 0.5, -0.05, 0.0], 
                                   bounds=([1, 0, -1, -1], [18, 1, 1, 1]))
        estimated_taus.append(popt_binned[0])
    except Exception as e:
        print(f"Fit failed for {b}-min bin: {e}")
        estimated_taus.append(np.nan)

# Plot out the Deviation Chart

In [ ]:
plt.figure(figsize=(10, 6))

# The Main Line Plot showing the drift
plt.plot(bin_labels,
         estimated_taus,
         marker='o',
         markersize=12,
         color='crimson', 
         linewidth=3,
         markeredgecolor='black',
         zorder=3)

# Add a horizontal dashed line representing the 'True Continuous' baseline
baseline_tau = estimated_taus[0]
plt.axhline(y=baseline_tau,
            color='gray',
            linestyle='--',
            linewidth=2, 
            label=f'True Continuous $\\tau$ ({baseline_tau:.2f} min)', zorder=1)

# Annotate each point with its specific Tau value
for i, tau_val in enumerate(estimated_taus):
    if not np.isnan(tau_val):
        plt.annotate(f"{tau_val:.2f} min", 
                     (i, tau_val), 
                     textcoords="offset points", 
                     xytext=(0,15), 
                     ha='center', 
                     fontsize=11, 
                     fontweight='bold')

# Labelling
plt.title('Distortion of Estimated Change Point ($\\tau$)', fontsize=15, pad=15)
plt.xlabel('Data Aggregation Level (Bin Size)', fontsize=13)
plt.ylabel('Estimated Change Point $\\tau$ (minutes)', fontsize=13)

# Dynamically set Y-axis limits so annotations aren't cut off
valid_taus = [t for t in estimated_taus if not np.isnan(t)]
if valid_taus:
    plt.ylim(max(0, min(valid_taus) - 2), max(valid_taus) + 3)

plt.grid(axis='both', linestyle=':', alpha=0.6)
plt.legend(fontsize=12, loc='upper left')
plt.tight_layout()

plt.show()